# Mental Health in the Tech Industry
## CS 210: Data Management for Data Science
**Authors:** Vanij Patel and Sean Lumasag

**Research Question:** What workplace and personal factors predict whether a tech worker will seek mental health treatment, and how does mental health interfere with their work?

**Datasets:** OSMI Mental Health in Tech Surveys — 2014 (1,259 rows) and 2016 (1,433 rows)

In [1]:
import sys
print(sys.executable)
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, f1_score, accuracy_score)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

os.makedirs('output_figures', exist_ok=True)

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#2ca02c'

print("All libraries loaded.")

/usr/local/bin/python3
All libraries loaded.


In [4]:
df14 = pd.read_csv('2014.csv')
df16 = pd.read_csv('2016.csv')

connect = sqlite3.connect('mental_health.db')
df14.to_sql('mental_health_2014', connect, if_exists='replace', index=False)
df16.to_sql('mental_health_2016', connect, if_exists='replace', index=False)
print("Data loaded into SQLite database.")
print(f"2014: {df14.shape[0]} rows, {df14.shape[1]} columns")
print(f"2016: {df16.shape[0]} rows, {df16.shape[1]} columns")
print("\nSaved to 'mental_health.db'.")

Data loaded into SQLite database.
2014: 1259 rows, 27 columns
2016: 1433 rows, 63 columns

Saved to 'mental_health.db'.


In [14]:
null14 = (df14.isnull().sum() / len(df14) * 100).round(1)
print("2014 Columns with >20% Nulls:")
print(null14[null14 > 20].sort_values(ascending=False))

2014 Columns with >20% Nulls:
comments          87.0
state             40.9
work_interfere    21.0
dtype: float64


In [9]:
null16 = (df16.isnull().sum() / len(df16) * 100).round(1)
print("2016 Columns with >20% Nulls:")
print(null16[null16 > 20].sort_values(ascending=False))

2016 Columns with >20% Nulls:
If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?                                                        90.0
If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?                                                85.8
Is your primary role within your company related to tech/IT?                                                                                                                        81.6
Do you know local or online resources to seek help for a mental health disorder?                                                                                                    80.0
If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?                                                        80.0
Do you have medical coverage (private insuran

In [15]:
print("2014 columns:", list(df14.columns))
print(f"\n2016 has {len(df16.columns)} columns written as longer questions.")
print("Example 2016 column:", df16.columns[0])

2014 columns: ['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed', 'family_history', 'treatment', 'work_interfere', 'no_employees', 'remote_work', 'tech_company', 'benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'leave', 'mental_health_consequence', 'phys_health_consequence', 'coworkers', 'supervisor', 'mental_health_interview', 'phys_health_interview', 'mental_vs_physical', 'obs_consequence', 'comments']

2016 has 63 columns written as longer questions.
Example 2016 column: Are you self-employed?
